In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("read-iceberg")
    .master("local[*]")

    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")

    # Iceberg
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config(
        "spark.sql.catalog.spark_catalog.type",
        "hadoop"
    )
    .config(
        "spark.sql.catalog.spark_catalog.warehouse",
        "s3a://chinook-lake"
    )

    # MinIO
    .config(
        "spark.hadoop.fs.s3a.endpoint",
        "http://localhost:9000"
    )
    .config(
        "spark.hadoop.fs.s3a.access.key",
        "admin"
    )
    .config(
        "spark.hadoop.fs.s3a.secret.key",
        "admin123"
    )
    .config(
        "spark.hadoop.fs.s3a.path.style.access",
        "true"
    )
    .config(
        "spark.hadoop.fs.s3a.connection.ssl.enabled",
        "false"
    )
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider"
    )
    .config(
        "spark.hadoop.fs.s3a.signing.algorithm",
        "S3V4"
    )

    # ВАЖНО: версия должна совпадать с Hadoop в Spark
    .config(
        "spark.jars.packages",
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.6.1,"
        "org.apache.hadoop:hadoop-aws:3.3.4"
    )

    .getOrCreate()
)

In [2]:
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()

print("endpoint:", hadoop_conf.get("fs.s3a.endpoint"))
print("access key:", hadoop_conf.get("fs.s3a.access.key"))
print("provider:", hadoop_conf.get("fs.s3a.aws.credentials.provider"))
print("signing:", hadoop_conf.get("fs.s3a.signing.algorithm"))

endpoint: http://localhost:9000
access key: admin
provider: org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider
signing: S3V4


In [5]:
spark.sql("SHOW TABLES IN silver").show()

+---------+----------+-----------+
|namespace| tableName|isTemporary|
+---------+----------+-----------+
|   silver|artist_sat|      false|
|   silver|artist_hub|      false|
+---------+----------+-----------+

